# ATML PA4 — full run on Kaggle (2 x T4)

Runs Tasks 1, 2 and 3 end to end with the code in `danishrasool10/ATML-PA-4` and the course data in `AbDu11aHHH/atml-pa4-datasets`.

**Before you press Run (notebook settings, right-hand panel):**
1. *Session options → Accelerator*: **GPU T4 x2**. *Internet*: **On**. *Persistence*: Files only (default).
2. *Add-ons → Secrets*: add **`HF_TOKEN`** (a Hugging Face read token; turn *Notebook access* on for it). Optional: **`GH_TOKEN`** (GitHub token, only needed if the code repo is private or you want to push results back).
3. Run cells top to bottom (or *Save Version → Save & Run All (Commit)* to run unattended; a Kaggle session lasts at most 12 h, so the runner stops starting new jobs after `DEADLINE_HOURS` and can be resumed — see the last cells).

T4 has no bf16: everything uses FP16 (as the assignment specifies). Everything is resumable: re-running the run cell skips finished jobs.

In [ ]:
import os, subprocess, sys, shutil, json, time
import torch
n = torch.cuda.device_count()
print("GPUs:", n, [torch.cuda.get_device_name(i) for i in range(n)])
print(subprocess.run("nvidia-smi --query-gpu=name,memory.total --format=csv", shell=True, capture_output=True, text=True).stdout)
assert n >= 1, "Turn on a GPU accelerator (T4 x2) in the notebook settings"
if n < 2:
    print("WARNING: only one GPU visible; the scheduler will serialise jobs (set GPUS = '0').")

## 1. Configuration

In [ ]:
WORK = "/kaggle/working"
CODE_URL = "https://github.com/danishrasool10/ATML-PA-4.git"
CODE_BRANCH = "task1-task2-ab"       # change to "main" once the pull request is merged
DATA_URL = "https://github.com/AbDu11aHHH/atml-pa4-datasets.git"
GPUS = ",".join(str(i) for i in range(min(2, n)))   # "0,1"
DEADLINE_HOURS = 11.0                # stop launching jobs after this (Kaggle hard limit is 12 h); running jobs get a 15 min grace
RESTORE_DIR = ""                     # e.g. "/kaggle/input/<previous-notebook-output>/ATML-PA-4/results" to continue an earlier session
PUSH_RESULTS = False                 # True: push results/ to branch 'results-kaggle' (needs GH_TOKEN secret with write access)
RUN_SMOKE = False                    # True: first run the whole suite on tiny models (CPU, ~10 min) to check the environment
REPO = f"{WORK}/ATML-PA-4"
DATA = f"{WORK}/atml-pa4-datasets"

## 2. Secrets, clone, dependencies

In [ ]:
secrets = {}
try:
    from kaggle_secrets import UserSecretsClient
    us = UserSecretsClient()
    for k in ("HF_TOKEN", "GH_TOKEN"):
        try:
            secrets[k] = us.get_secret(k)
        except Exception as e:
            print(f"secret {k}: not set")
except ImportError:
    print("not on Kaggle: using environment variables")
    secrets = {k: os.environ[k] for k in ("HF_TOKEN", "GH_TOKEN") if k in os.environ}
if "HF_TOKEN" in secrets:
    os.environ["HF_TOKEN"] = os.environ["HUGGING_FACE_HUB_TOKEN"] = secrets["HF_TOKEN"]
print("HF token set:", "HF_TOKEN" in secrets)

def clone(url, dest, branch=None):
    auth = url.replace("https://", f"https://x-access-token:{secrets['GH_TOKEN']}@") if "GH_TOKEN" in secrets else url
    if os.path.isdir(dest + "/.git"):
        subprocess.run(["git", "-C", dest, "fetch", "-q", "origin"], check=True)
        subprocess.run(["git", "-C", dest, "checkout", "-q", branch or "HEAD"], check=True)
        subprocess.run(["git", "-C", dest, "pull", "-q", "--ff-only", auth] + ([branch] if branch else []), check=True)
    else:
        cmd = ["git", "clone", "-q"] + (["-b", branch] if branch else []) + [auth, dest]
        subprocess.run(cmd, check=True)
        subprocess.run(["git", "-C", dest, "remote", "set-url", "origin", url], check=True)   # do not keep the token in .git/config

clone(CODE_URL, REPO, CODE_BRANCH)
clone(DATA_URL, DATA)
print(subprocess.run(f"git -C {REPO} log --oneline -3; git -C {DATA} log --oneline -1", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# keep Kaggle's CUDA build of torch; install everything else (pins transformers==4.57.1, huggingface_hub==0.36.2 for Part C)
tv = subprocess.run([sys.executable, "-c", "import torch;print(torch.__version__.split('+')[0])"], capture_output=True, text=True).stdout.strip()
open("/tmp/constraints.txt", "w").write(f"torch=={tv}\n")
r = subprocess.run(f"cd {REPO} && pip install -q -c /tmp/constraints.txt -r requirements.txt && pip install -q -c /tmp/constraints.txt -r {DATA}/requirements-public.txt",
                   shell=True, capture_output=True, text=True)
print(r.stdout[-1500:], r.stderr[-1500:])
print(subprocess.run([sys.executable, "-c", "import torch,transformers,huggingface_hub as h;print('torch',torch.__version__,'transformers',transformers.__version__,'hub',h.__version__)"], capture_output=True, text=True).stdout)
from huggingface_hub import login
if "HF_TOKEN" in secrets:
    login(token=secrets["HF_TOKEN"], add_to_git_credential=False)

In [ ]:
# optional: continue a previous session (copies finished results + job markers back)
if RESTORE_DIR and os.path.isdir(RESTORE_DIR):
    shutil.copytree(RESTORE_DIR, f"{REPO}/results", dirs_exist_ok=True)
    print("restored", RESTORE_DIR, "->", f"{REPO}/results")
    print(subprocess.run(f"ls {REPO}/results/.jobs | head -50", shell=True, capture_output=True, text=True).stdout)

## 3. Optional: plumbing check of the whole suite on tiny models (no real results)

In [ ]:
if RUN_SMOKE:
    !cd {REPO} && python -u scripts/kaggle_runner.py --repo . --data ../atml-pa4-datasets --smoke --gpus ""

## 4. The full run

`scripts/kaggle_runner.py` schedules the jobs on the two GPUs:

| phase | jobs |
|---|---|
| any order, 2 GPU lanes + 1 CPU slot | unit tests, `setup_data.py --public`, `--prefix-banks`, Task 1 (all stages), Task 3 validation gate → 4 distillation runs → prefix banks + standalone quality, Part C evaluation, MQAR B1/B2/B3 |
| quiet phase (machine otherwise idle, GPU 0) | Part A attention benchmark, MQAR inference sweep, Part C packed-cache benchmark, Task 3 speculative-decoding benchmarks + fixed-64 throughput, then the analysis jobs |

Logs: `results/logs/<job>.log`. Status of every job: `results/.jobs/summary.json`. The cell prints progress; interrupting the cell (or the session ending) is safe — run it again to resume.

In [ ]:
!cd {REPO} && python -u scripts/kaggle_runner.py --repo . --data ../atml-pa4-datasets --gpus {GPUS} --deadline-hours {DEADLINE_HOURS}

## 5. What finished / what failed

In [ ]:
p = f"{REPO}/results/.jobs/summary.json"
if os.path.exists(p):
    s = json.load(open(p))
    for k, v in s.items():
        print(f"{v:12s} {k}")
    todo = [k for k, v in s.items() if v != "done"]
    print("\nNOT DONE:", todo or "nothing")
    for k in todo:
        lf = f"{REPO}/results/logs/{k}.log"
        if os.path.exists(lf):
            print(f"\n--- {k} (last lines) ---\n" + "".join(open(lf).readlines()[-15:]))

## 6. Package the results

Writes `/kaggle/working/pa4_results.zip` (results, logs, environment metadata; model checkpoints and raw data are not included). After the session: **Save Version** so the output is kept, then download the zip from the notebook's *Output* tab. To continue an unfinished run in a new session, add this notebook's output as an *Input* and set `RESTORE_DIR` in the configuration cell.

In [ ]:
os.chdir(WORK)
shutil.make_archive(f"{WORK}/pa4_results", "zip", REPO, "results")
print(subprocess.run(f"du -sh {WORK}/pa4_results.zip; du -sh {REPO}/results/*/ | sort -h | tail", shell=True, capture_output=True, text=True).stdout)

In [ ]:
# optional: push results/ to a separate branch (never touches the code branch). Needs GH_TOKEN with repo write access.
if PUSH_RESULTS:
    assert "GH_TOKEN" in secrets, "add a GH_TOKEN secret first"
    auth = CODE_URL.replace("https://", f"https://x-access-token:{secrets['GH_TOKEN']}@")
    sh = f'''cd {REPO} && git checkout -B results-kaggle && git add -f results && git -c user.name=kaggle-run -c user.email=noreply@example.com commit -qm "Kaggle 2xT4 run results" && git push -q {auth} results-kaggle --force'''
    print(subprocess.run(sh, shell=True, capture_output=True, text=True).stderr.replace(secrets["GH_TOKEN"], "***")[-800:])